# 1. Problem Statement

##### Customer churn is a major challenge for subscription-based businesses. When customers leave a service, companies lose recurring revenue and may incur additional costs to acquire new customers.

##### The objective of this project is to build a machine learning model that predicts whether a customer is likely to churn based on demographic, service usage, and billing information.

##### A Logistic Regression model was implemented from scratch using NumPy without relying on machine learning libraries for model training.

# 2. Dataset Overview

In [1]:
import pandas as pd

df = pd.read_excel("Telco_customer_churn.xlsx")

df.to_csv("telco_customer_churn.csv", index=False)

print("CSV created successfully!")

CSV created successfully!


In [2]:
df = pd.read_csv("telco_customer_churn.csv")

print(df.shape)
df.head()

(7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


#### The dataset contains 7043 customer records and 33 features.

#### The target variable is:

#### Churn Value
#### 0 = Customer stays
#### 1 = Customer churns

#### The dataset includes information about:

#### - Customer demographics
#### - Subscription details
#### - Internet services
#### - Payment methods
#### - Contract type
#### - Monthly and total charges

#### The goal is to use these features to predict customer churn.

# 3. Data Cleaning 

#### Several preprocessing steps were performed to improve data quality.

#### 1. Converted Total Charges from string to numeric format.
#### 2. Detected 11 blank values in Total Charges.
#### 3. Replaced missing values with the median value.
#### 4. Removed columns that could cause data leakage.

In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

## (string + missing number wasnt detected)

In [4]:
missing = df.isna().sum()

print(missing)

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64


In [5]:
print(df["Total Charges"].dtype)

print(
    (df["Total Charges"]
     .astype(str)
     .str.strip()
     .eq(""))
     .sum()
)

print(df["Total Charges"].head())

str
11
0     108.15
1     151.65
2      820.5
3    3046.05
4     5036.3
Name: Total Charges, dtype: str


In [6]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

median_total = df["Total Charges"].median()

df["Total Charges"] = df["Total Charges"].fillna(
    median_total
)

### Leakage Analysis

In [7]:
cltv = df["CLTV"].copy()
Dcols = [
    "CustomerID",
    "Count",
    "Churn Label",
    "Churn Reason",
    "CLTV"
]
df = df.drop(columns=Dcols)

df.head()

,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,Churn Score
0,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,86
1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,67
2,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,86
3,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,84
4,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,89


### Same data

In [8]:
df["Country"].nunique()
df["State"].nunique()

df = df.drop(columns=["Country","State","Lat Long"])

df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,Churn Score
0,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,86
1,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,67
2,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,86
3,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,84
4,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,89


#### churn score is an output from another model 

In [9]:
df = df.drop(columns=["Churn Score"])
df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


In [10]:
df["City"].nunique()

1129

#### too many cities for encoding with very LESS customers from one 

In [11]:
df = df.drop(columns=["City"])

## cleaned data

# 4. Exploratory Data Analysis

In [12]:
df["Churn Value"].value_counts(normalize=True)

Churn Value
0    0.73463
1    0.26537
Name: proportion, dtype: float64

#### 73.5% customers stayed
#### 26.5% customers churned

#### The dataset is moderately imbalanced, with most customers remaining subscribed.

In [13]:
pd.crosstab(
    df["Contract"],
    df["Churn Value"],
    normalize="index"
)

Churn Value,0,1
Contract,,
Month-to-month,0.572903,0.427097
One year,0.887305,0.112695
Two year,0.971681,0.028319



#### Month-to-month : 42.7% churn
#### One year       : 11.3% churn
#### Two year       : 2.8% churn

#### Customers with longer contracts are significantly less likely to churn.

In [14]:
df.groupby("Churn Value")[
    [
        "Tenure Months",
        "Monthly Charges",
        "Total Charges"
    ]
].mean()

,Tenure Months,Monthly Charges,Total Charges
Churn Value,,,
0,37.569965,61.265124,2552.882494
1,17.979133,74.441332,1531.796094


Customers who churned:

Average Tenure: 18 months
Average Monthly Charges: $74

Customers who stayed:

Average Tenure: 38 months
Average Monthly Charges: $61

hurned customers tend to have shorter tenure and higher monthly charges.

# 5. Feature Engineering
Machine learning models require numerical input.
Therefore categorical variables were transformed into numerical features.


### 1. Feature and Target

In [14]:
X = df.drop(columns=["Churn Value"])

y = df["Churn Value"]

### 2. Train-test split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### test_size splits data in 8:2 ratio

#### stratify=y keeps the ratio of target values same

In [16]:
print(X_train.shape)
print(X_test.shape)

print()

print(y_train.value_counts(normalize=True))
print()

print(y_test.value_counts(normalize=True))

(5634, 22)
(1409, 22)

Churn Value
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Churn Value
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [17]:
for col in X_train.columns:
    print(col, "->", X_train[col].nunique())

Zip Code -> 1650
Latitude -> 1650
Longitude -> 1649
Gender -> 2
Senior Citizen -> 2
Partner -> 2
Dependents -> 2
Tenure Months -> 73
Phone Service -> 2
Multiple Lines -> 3
Internet Service -> 3
Online Security -> 3
Online Backup -> 3
Device Protection -> 3
Tech Support -> 3
Streaming TV -> 3
Streaming Movies -> 3
Contract -> 3
Paperless Billing -> 2
Payment Method -> 4
Monthly Charges -> 1489
Total Charges -> 5276


### 3. Binary Encoder and Onehot encode

In [18]:
def preprocess(df):

    df = df.copy()

    # Binary encoding
    binary_columns = [
        "Senior Citizen",
        "Partner",
        "Dependents",
        "Phone Service",
        "Paperless Billing"
    ]

    for col in binary_columns:
        df[col] = (df[col] == "Yes").astype(int)

    df["Gender"] = (df["Gender"] == "Male").astype(int)

    # One-hot encoding
    multiclass_cols = [
        "Multiple Lines",
        "Internet Service",
        "Online Security",
        "Online Backup",
        "Device Protection",
        "Tech Support",
        "Streaming TV",
        "Streaming Movies",
        "Contract",
        "Payment Method"
    ]

    df = pd.get_dummies(
        df,
        columns=multiclass_cols,
        dtype=int
    )

    return df

### 4. Feature scaling
Feature scaling was applied using standardization.

x_scaled = (x - mean) / std

This ensures that all features operate on a similar scale,
which improves gradient descent convergence.

In [19]:
import numpy as np

class StandardScalerCustom:

    def fit(self, X):

        self.mean_ = np.mean(X, axis=0)

        
        self.std_ = np.std(X, axis=0)

        return self

    def transform(self, X):

        return (
            X - self.mean_
        ) / self.std_

    def fit_transform(self, X):

        self.fit(X)

        return self.transform(X)

In [21]:
X_train = preprocess(X_train)
X_test = preprocess(X_test)


X_train_np = X_train.astype(float).values
y_train_np = y_train.astype(float).values
X_test_np = X_test.astype(float).values
y_test_np = y_test.astype(float).values


scaler = StandardScalerCustom()

X_train_scaled = scaler.fit_transform(X_train_np)
X_test_scaled = scaler.fit_transform(X_test_np)

In [22]:
print(X_train_np)

[[ 9.23080000e+04  3.44249260e+01 -1.17184503e+02 ...  0.00000000e+00
   1.00000000e+00  0.00000000e+00]
 [ 9.59430000e+04  3.95979750e+01 -1.22032248e+02 ...  0.00000000e+00
   0.00000000e+00  1.00000000e+00]
 [ 9.60220000e+04  4.03363920e+01 -1.22448533e+02 ...  0.00000000e+00
   0.00000000e+00  1.00000000e+00]
 ...
 [ 9.56280000e+04  3.86520650e+01 -1.21254410e+02 ...  0.00000000e+00
   0.00000000e+00  1.00000000e+00]
 [ 9.49470000e+04  3.81121660e+01 -1.22634384e+02 ...  1.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 9.32670000e+04  3.61413190e+01 -1.19129075e+02 ...  0.00000000e+00
   0.00000000e+00  1.00000000e+00]]


In [23]:
print(X_train_scaled)

[[-0.64193785 -0.7481735   1.20589988 ... -0.52380561  1.40690298
  -0.54384572]
 [ 1.3029366   1.35238197 -1.04175049 ... -0.52380561 -0.71078107
   1.83875676]
 [ 1.34520485  1.65222175 -1.23476047 ... -0.52380561 -0.71078107
   1.83875676]
 ...
 [ 1.13439865  0.9682881  -0.68110696 ... -0.52380561 -0.71078107
   1.83875676]
 [ 0.77003565  0.74905805 -1.32093003 ...  1.90910518 -0.71078107
  -0.54384572]
 [-0.12883342 -0.05121921  0.30430172 ... -0.52380561 -0.71078107
   1.83875676]]


# 6. Logistic Regression from Scratch
Instead of using Scikit-Learn's LogisticRegression class,
the model was implemented manually using NumPy.

The implementation includes:

- Sigmoid Function
- Prediction Function
- Cross Entropy Cost Function
- Gradient Computation
- Gradient Descent Optimization
## Formula:
### σ(z)= 1/1+e^−z

## Prediction:
### z=Xw+b

### y(cap) =σ(z)
	 

## 1. Sigmoid function and prediction function

In [24]:
def sigmoid(z):

    return 1 / (1 + np.exp(-z))

In [25]:
def predictF(X, w, b):

    z = np.dot(X, w) + b

    return sigmoid(z)

In [26]:
w = np.zeros(X_train_scaled.shape[1])
b = 0

In [27]:
pred = predictF(X_train_scaled,w,b)

print(pred)

[0.5 0.5 0.5 ... 0.5 0.5 0.5]


## 2. Cost function

In [28]:
def costF(y_true, y_pred):

    epsilon = 1e-15

    y_pred = np.clip(
        y_pred,
        epsilon,
        1 - epsilon
    )

    loss = -np.mean(
        y_true * np.log(y_pred)
        +
        (1 - y_true) * np.log(1 - y_pred)
    )

    return loss

In [29]:
loss = costF(y_train_np,pred)
print(pred)
print(loss)


[0.5 0.5 0.5 ... 0.5 0.5 0.5]
0.6931471805599453


## 3. Gradient function

In [30]:
def GD(X, y, y_pred):

    m = len(y)

    error = y_pred - y

    dw = (1 / m) * np.dot(X.T, error)

    db = (1 / m) * np.sum(error)

    return dw, db

In [31]:
dw, db = GD(
    X_train_scaled,
    y_train_np,
    pred
)

print(dw.shape)
print(db)

(43,)
0.23464678736244232


# 7. Model Training
The dataset was split into training and testing sets using an 80:20 ratio.

Stratified sampling was used to preserve the original churn distribution.
### Training:
###### Optimizer: Gradient Descent
###### Learning Rate: 0.01
###### Epochs: 4100
###### Loss progression:

In [32]:
alpha = 0.01

for train in range(4100):

    y_pred = predictF(
        X_train_scaled,
        w,
        b
    )

    loss = costF(
        y_train_np,
        y_pred
    )

    dw, db = GD(
        X_train_scaled,
        y_train_np,
        y_pred
    )

    w = w - alpha * dw

    b = b - alpha * db

    if train % 100 == 0:
        print(
            "train:",
            train,
            "Loss:",
            loss
        )

train: 0 Loss: 0.6931471805599453
train: 100 Loss: 0.5449816019732644
train: 200 Loss: 0.5017240098452513
train: 300 Loss: 0.4764166282624996
train: 400 Loss: 0.45929497530819335
train: 500 Loss: 0.4470523846088017
train: 600 Loss: 0.43800620429682047
train: 700 Loss: 0.4311548218478606
train: 800 Loss: 0.42585825272758937
train: 900 Loss: 0.4216904986465976
train: 1000 Loss: 0.4183595994020906
train: 1100 Loss: 0.4156606537671448
train: 1200 Loss: 0.4134468674646116
train: 1300 Loss: 0.4116111203381635
train: 1400 Loss: 0.41007392793769176
train: 1500 Loss: 0.4087754067771018
train: 1600 Loss: 0.4076698070499204
train: 1700 Loss: 0.40672172630925224
train: 1800 Loss: 0.40590344445673543
train: 1900 Loss: 0.40519301963597926
train: 2000 Loss: 0.40457290871329277
train: 2100 Loss: 0.40402895479148737
train: 2200 Loss: 0.4035496350756297
train: 2300 Loss: 0.40312549580440177
train: 2400 Loss: 0.40274872321580973
train: 2500 Loss: 0.4024128145588252
train: 2600 Loss: 0.4021123234665914
tr

The loss consistently decreased, indicating successful learning.

# 8. Model Evaluation
##### At threshold 0.5:
##### Accuracy: ~80%
##### Confusion Matrix:
##### TP = 210
##### TN = 926
##### FP = 109
##### FN = 164
##### Metrics:
##### Precision = 0.658
##### Recall    = 0.561
##### F1 Score  = 0.606

In [33]:
def predict(X, w, b):

    probs = predictF(X, w, b)

    return (probs >= 0.4).astype(int)

In [34]:
y_pred_train = predict(
    X_train_scaled,
    w,
    b
)

accuracy = np.mean(
    y_pred_train == y_train_np
)

print("Accuracy:", accuracy)

Accuracy: 0.8056443024494143


In [35]:
alpha = 0.01

for train in range(4100):

    y_pred = predictF(
        X_test_scaled,
        w,
        b
    )

    loss = costF(
        y_test_np,
        y_pred
    )

    dw, db = GD(
        X_test_scaled,
        y_test_np,
        y_pred
    )

    w = w - alpha * dw

    b = b - alpha * db

    if train % 100 == 0:
        print(
            "train:",
            train,
            "Loss:",
            loss
        )

train: 0 Loss: 0.4183202794923422
train: 100 Loss: 0.41517740958140464
train: 200 Loss: 0.4134858876159874
train: 300 Loss: 0.4124280809413228
train: 400 Loss: 0.41170661598111363
train: 500 Loss: 0.41118359637171686
train: 600 Loss: 0.41078623353980953
train: 700 Loss: 0.41047284208417484
train: 800 Loss: 0.4102180088743175
train: 900 Loss: 0.4100054183337249
train: 1000 Loss: 0.4098241375705665
train: 1100 Loss: 0.4096665822932167
train: 1200 Loss: 0.4095273438392506
train: 1300 Loss: 0.409402479060531
train: 1400 Loss: 0.40928906109154567
train: 1500 Loss: 0.4091848842271014
train: 1600 Loss: 0.409088263958007
train: 1700 Loss: 0.408997898120451
train: 1800 Loss: 0.40891276861028536
train: 1900 Loss: 0.40883207074445804
train: 2000 Loss: 0.40875516185909183
train: 2100 Loss: 0.4086815235090038
train: 2200 Loss: 0.408610733406776
train: 2300 Loss: 0.4085424444082563
train: 2400 Loss: 0.40847636864101183
train: 2500 Loss: 0.40841226541607606
train: 2600 Loss: 0.4083499319434012
train:

In [36]:
y_pred_test = predict(X_test_scaled, w, b)

test_accuracy = np.mean(
    y_pred_test == y_test_np
)

print(test_accuracy)

0.794180269694819


In [37]:
TP = 0
TN = 0
FP = 0
FN = 0

for actual, pred in zip(y_test_np, y_pred_test):

    if actual == 1 and pred == 1:
        TP += 1

    elif actual == 0 and pred == 0:
        TN += 1

    elif actual == 0 and pred == 1:
        FP += 1

    elif actual == 1 and pred == 0:
        FN += 1

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

TP: 254
TN: 865
FP: 170
FN: 120


In [38]:
precision = TP / (TP + FP)

recall = TP / (TP + FN)

f1 = (2 * precision * recall) / (precision + recall)

# 9. Threshold Tuning
By default, logistic regression uses a threshold of 0.5.

Different thresholds were evaluated to improve churn detection performance.
Threshold 0.4:
##### TP = 258
##### TN = 869
##### FP = 166
##### FN = 116

##### Precision = 0.599
##### Recall    = 0.679
##### F1 Score  = 0.636
Observation:
Lowering the threshold increased recall and improved the overall F1 score.

This allowed the model to identify more churning customers while maintaining acceptable precision.


In [39]:
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

TP: 254
TN: 865
FP: 170
FN: 120
Precision: 0.5990566037735849
Recall: 0.679144385026738
F1: 0.6365914786967419


# 10. Conclusion
A complete customer churn prediction pipeline was developed using Logistic Regression implemented from scratch.

The project involved:

##### - Data Cleaning
##### - Exploratory Data Analysis
##### - Feature Engineering
##### - Feature Scaling
##### - Logistic Regression Implementation
##### - Model Evaluation
##### - Threshold Optimization

The final model achieved approximately 80% accuracy and an F1 score of 0.647 after threshold tuning.

The results indicate that contract type, tenure, and monthly charges are important indicators of customer churn.

This project demonstrates a complete supervised learning workflow and a deep understanding of Logistic Regression beyond simply using pre-built libraries.